In [1]:
import pandas as pd

# Simulated e-commerce dataset
data = {
    "product_id": [101, 102, 103, 104],
    "product_name": ["Running Shoes Ultra", "Noise Cancelling Headphones", "Cotton T-Shirt", "Gaming Mouse"],
    "review_text": [
        "These shoes are incredibly comfortable for long distance running. The arch support is phenomenal, but they run a bit small.",
        "The active noise cancellation on these is top-tier. I use them on flights and can't hear a thing. Battery lasts for days.",
        "A standard, comfortable oversized t-shirt. The cotton is breathable, but the color faded slightly after the first wash.",
        "Extremely responsive and lightweight. The RGB lighting is customizable, perfect for competitive gaming setups."
    ]
}
df = pd.DataFrame(data)

print("--- Initial Data ---")
df.dropna(subset=['review_text'], inplace=True)
display(df)

--- Initial Data ---


C:\Users\prabh\AppData\Local\Temp\ipykernel_29076\1260429562.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


,product_id,product_name,review_text
0,101,Running Shoes Ultra,These shoes are incredibly comfortable for lon...
1,102,Noise Cancelling Headphones,The active noise cancellation on these is top-...
2,103,Cotton T-Shirt,"A standard, comfortable oversized t-shirt. The..."
3,104,Gaming Mouse,Extremely responsive and lightweight. The RGB ...


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,  
    chunk_overlap=20,
    separators=["\n\n", "\n", ".", " "]
)

def chunk_row(row):
    chunks = text_splitter.split_text(row['review_text'])
    return [{"product_id": row['product_id'], "chunk_text": chunk} for chunk in chunks]

chunked_data = []
for index, row in df.iterrows():
    chunked_data.extend(chunk_row(row))

df_chunks = pd.DataFrame(chunked_data)
print(f"Total chunks: {len(df_chunks)}")

Total chunks: 4


In [3]:
!pip install -U langchain langchain-text-splitters


  Using cached langchain-1.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached langchain-1.3.2-py3-none-any.whl (121 kB)
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.1
    Uninstalling langchain-1.3.1:
      Successfully uninstalled langchain-1.3.1


In [ ]:
import google.generativeai as genai
import os
import time
from dotenv import load_dotenv
load_dotenv(override=True)
from sentence_transformers import SentenceTransformer

load_dotenv()
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

embeddings = []
used_model = "None"

try:
    print("Attempting to use Google Gemini for embeddings...")
    for text in df_chunks['chunk_text']:
        result = genai.embed_content(
            model="models/embedding-001",
            content=text,
            task_type="retrieval_document",
        )
        embeddings.append(result['embedding'])
        time.sleep(0.5)
    used_model = "Gemini (768d)"
    
except Exception as e:
    print(f"Gemini failed ({e}).\nFalling back to local Hugging Face model...")
    hf_model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = hf_model.encode(df_chunks['chunk_text'].tolist()).tolist()
    used_model = "Hugging Face Local (384d)"

df_chunks['embedding'] = embeddings
vector_dimension = len(embeddings[0])

print(f"\nSuccess! Used: {used_model}")
print(f"Detected Vector Dimension: {vector_dimension}")

Attempting to use Google Gemini for embeddings...
Gemini failed (400 API key not valid. Please pass a valid API key. [reason: "API_KEY_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "generativelanguage.googleapis.com"
}
, locale: "en-US"
message: "API key not valid. Please pass a valid API key."
]).
Falling back to local Hugging Face model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Success! Used: Hugging Face Local (384d)
Detected Vector Dimension: 384


In [ ]:
from sqlalchemy import create_engine, text

DB_URL = os.getenv("DATABASE_URL")
engine = create_engine(DB_URL)

with engine.begin() as conn:
    conn.execute(text("CREATE EXTENSION IF NOT EXISTS vector;"))
    conn.execute(text("DROP TABLE IF EXISTS product_reviews;"))
    
    # Dynamically set the vector size based on the successful model
    conn.execute(text(f"""
        CREATE TABLE product_reviews (
            id SERIAL PRIMARY KEY,
            product_id INT,
            chunk_text TEXT,
            embedding vector({vector_dimension}) 
        );
    """))

print(f"Schema created with {vector_dimension} dimensions. Pushing data...")

with engine.begin() as conn:
    for _, row in df_chunks.iterrows():
        embedding_str = f"[{','.join(map(str, row['embedding']))}]"
        
        insert_query = text("""
            INSERT INTO product_reviews (product_id, chunk_text, embedding)
            VALUES (:pid, :text, :embed)
        """)
        conn.execute(insert_query, {
            "pid": row['product_id'], 
            "text": row['chunk_text'], 
            "embed": embedding_str
        })

print("Data successfully ingested!")